In [1]:
from dh_tool import create_daily_folder

In [2]:
today_dir = create_daily_folder()

In [ ]:
today_dir

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Serializer(Protocol):
    def serialize(self, data: dict) -> str:
        pass

# @runtime_checkable
class JSONSerializer:
    def serialize(self, data: dict) -> str:
        return "json"

print(isinstance(JSONSerializer(), Serializer))  # True


In [12]:
import threading
class EventBus:
    _instance = None  # 단일 인스턴스를 저장할 변수
    _lock = threading.Lock()  # 스레드 안전성을 위한 Lock 객체

    def __new__(cls):
        with cls._lock:  # 여러 스레드에서 동시에 접근하는 걸 방지
            if cls._instance is None:
                cls._instance = super().__new__(cls)  # 새로운 인스턴스 생성
                # cls._instance._events = {}  # 이벤트 저장용 딕셔너리 초기화
        return cls._instance  # 기존 인스턴스 반환

    def __init__(self):
        self._events  = {}
        pass

In [ ]:
bus1 = EventBus()
bus2 = EventBus()

print(bus1 is bus2)  # True (같은 인스턴스임)


In [ ]:
bus1._instance._events

In [19]:
def get_cell_addresses(df, condition):
    """
    DataFrame과 조건을 받아 조건을 만족하는 셀의 주소(A1, B2 등)를 반환
    - df: pandas DataFrame
    - condition: 불리언 Series 또는 DataFrame
    """
    cells = []

    # ✅ 1. Series인 경우 (특정 컬럼에만 조건 적용)
    if isinstance(condition, pd.Series):
        col_name = condition.name                      # 조건이 적용된 컬럼 이름
        col_idx = df.columns.get_loc(col_name)         # 해당 컬럼의 인덱스 (0부터 시작)
        col_letter = chr(65 + col_idx)                 # A, B, C ... (엑셀 열 이름)

        # 조건이 True인 경우 해당 셀 주소 저장
        for row_idx, match in condition.items():
            if match:
                excel_row = row_idx + 2                # 헤더가 1행, 데이터는 2행부터 시작
                cell_ref = f"{col_letter}{excel_row}"  # 셀 주소 (예: B2)
                cells.append(cell_ref)

    # ✅ 2. DataFrame인 경우 (여러 컬럼에 조건 적용)
    elif isinstance(condition, pd.DataFrame):
        for row_idx, row in condition.iterrows():
            for col_idx, match in enumerate(row):
                if match:
                    col_letter = chr(65 + col_idx)
                    excel_row = row_idx + 2
                    cell_ref = f"{col_letter}{excel_row}"
                    cells.append(cell_ref)

    else:
        raise ValueError("Condition must be a Series or DataFrame")

    return cells


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Score": [85, 92, 78],
    "Grade": ["A", "A", "B"]
})

# get_cell_addresses(df, )
condition = df["Score"] > 80
get_cell_addresses(df, condition)

In [ ]:
df

In [ ]:
str_condition = df.applymap(lambda x: isinstance(x, str))
str_condition
get_cell_addresses(df, str_condition)

In [ ]:
str_condition

In [ ]:
condition.to_frame()

---

In [1]:
from dh_tool.log_tool.logger import Logger 
from dh_tool.log_tool.handlers.console_handler import get_console_handler
from dh_tool.log_tool.decorators import auto_logger
from dh_tool.log_tool.context_managers import log_block
logger = Logger("name")
logger.add_handler(get_console_handler())

@auto_logger(logger)
def add(a, b):
    return a + b

with log_block(logger, "Computation Block"):
    result = add(5, 10)
    logger.info(f"Final Result: {result}")

2025-02-04 00:18:28,877 - INFO - Start: Computation Block
2025-02-04 00:18:28,878 - INFO - Called add with args=(5, 10), kwargs={}
2025-02-04 00:18:28,878 - INFO - add returned 15
2025-02-04 00:18:28,879 - INFO - Final Result: 15
2025-02-04 00:18:28,879 - INFO - End: Computation Block


In [2]:
from dotenv import load_dotenv
load_dotenv()
import os
api_key = os.getenv("OENAI_API_KEY")

In [3]:
api_key

In [2]:
from dh_tool import load, save
from dh_tool.common import *
from dh_tool.llm_tool import LLMConfig, GPTModel
from dh_tool.log_tool import Logger, get_console_handler, auto_logger, log_async_block

logger = Logger("my_logger", level="DEBUG")
logger.add_handler(get_console_handler())
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
logger.info(f"API key: {api_key}")

config = LLMConfig(
    model="gpt-4o-mini",
    api_key=api_key,
    generation_params={
        "max_completion_tokens": 12,
        "hi": 10,
        "temperature": 0.9,
    },
)

gpt = GPTModel(config)


# async with log_async_block(logger, "Generating gpt response"):
#     await gpt.generate("웃긴 얘기좀", parsed=True)
@auto_logger(logger)
async def test_gpt(text):
    result = await gpt.generate(text, parsed=True)
    return result

await test_gpt("웃긴 얘기좀")

2025-02-04 01:09:34,891 - INFO - API key: sk-svcacct-USlZrDeyHjRw7RsPNWurT3BlbkFJ8AagIBVIqNXspxE8MB7X
2025-02-04 01:09:34,891 - INFO - API key: sk-svcacct-USlZrDeyHjRw7RsPNWurT3BlbkFJ8AagIBVIqNXspxE8MB7X
2025-02-04 01:09:34,922 - DEBUG - Called test_gpt with args=["'웃긴 얘기좀'"], kwargs={}
2025-02-04 01:09:34,922 - DEBUG - Called test_gpt with args=["'웃긴 얘기좀'"], kwargs={}
2025-02-04 01:09:35,536 - INFO - test_gpt returned ('물론이죠! 여기에 하나의 웃긴', CompletionUsage(completion_tokens=12, prompt_tokens=13, total_tokens=25, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
2025-02-04 01:09:35,536 - INFO - test_gpt returned ('물론이죠! 여기에 하나의 웃긴', CompletionUsage(completion_tokens=12, prompt_tokens=13, total_tokens=25, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0

('물론이죠! 여기에 하나의 웃긴',
 CompletionUsage(completion_tokens=12, prompt_tokens=13, total_tokens=25, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))